> **UNQUANTIZED RERUN** — generated copy of `21_e1b_fluency_pareto.ipynb` with
> `meta-llama/Llama-3.1-8B-Instruct` (bf16, sharded via `device_map="auto"`)
> in place of the AWQ-INT4 variant. All `resources/` outputs carry the
> `_unquantized` suffix; the pair selection (`15_e0_selected_pairs.json`) is
> shared with the quantized runs so results are directly comparable.
> Hardcoded reference values quoted in comments/markdown still describe the
> AWQ runs. Original notebook left untouched.


# 21 — E1b: the fluency dial + beam × aligned mean

Two E1 follow-ups in one factorial sweep, all under **beam search** (the E1-winning recipe), k=4 / steps=24 / 20 prompts per cell:

1. **Fluency-aware selection (λ sweep):** selection score = cumulative concept score + λ · cumulative token log-probability, λ ∈ {0, 0.1, 0.2, 0.4, 1, 2} (the playground showed the knee sits near λ≈1). Question: how much PPL does a unit of expression cost along the dial — can we get inside (or near) the 2.5× guardrail while keeping most of the steering?
2. **beam × aligned mean:** E2/E2b showed the aligned (GPA) mean beats the extrinsic mean under *greedy* decoding below ρ≈0.30. Does that stack with beam search? The λ=0 slice of {extrinsic, aligned} × strata answers it.

Grid: 2 means × 6 λ × 3 ρ strata = 36 cells, 720 records.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    "meta-llama/Llama-3.1-8B-Instruct",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24
LAMBDAS = (0.0, 0.1, 0.2, 0.4, 1.0, 2.0)
MEANS = ("extrinsic", "aligned")

SELECTION = json.loads(Path("resources/15_e0_selected_pairs.json").read_text())[
    "selected"
]
for stratum, info in SELECTION.items():
    print(f"{stratum:>6}: {info['a']} / {info['b']}  rho={info['rho']:.3f}")

In [ ]:
def chat(user: str, assistant: str = "") -> str:
    return (
        f"<|start_header_id|>user<|end_header_id|>{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{assistant}"
    )


PROMPTS = [
    chat("What men can be?", "1."),
    chat("What women can be?", "1."),
    chat("Tell me a short story.", "Once"),
    chat("Describe your favorite place.", "My favorite place is"),
    chat("What do you like to do on weekends?", "I like"),
    chat("Give me some advice for a good life.", "1."),
    chat(
        "What is the most important thing in the world?", "The most important thing is"
    ),
    chat("Describe the weather today.", "Today the weather is"),
    chat("What should I cook for dinner?", "You should"),
    chat("Name things that make people happy.", "1."),
    chat("What did you do today?", "Today I"),
    chat("Describe a memorable meal.", "The most memorable meal"),
    chat("What is a good gift idea?", "A good gift"),
    chat("Describe your hometown.", "My hometown is"),
    chat("Tell me about your favorite season.", "My favorite season is"),
    chat("What makes a good friend?", "A good friend"),
    chat("Describe a place you would like to visit.", "I would like to visit"),
    chat("What do you do to relax?", "To relax, I"),
    chat("Tell me something interesting.", "Here is something interesting:"),
    chat("Describe your morning routine.", "My morning starts"),
]

LOG_PATH = Path("resources/21_e1b_fluency_unquantized.jsonl")
LOG_PATH.unlink(missing_ok=True)

harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(PROMPTS, max_new_tokens=STEPS + 2)
print("baseline ready")

In [ ]:
for stratum, info in SELECTION.items():
    concept_a = fur.get_concept_cached(info["a"], MIN_LEMMAS, MAX_TOKENS)
    concept_b = fur.get_concept_cached(info["b"], MIN_LEMMAS, MAX_TOKENS)
    synsets = [info["a"], info["b"]]
    concepts = {info["a"]: concept_a, info["b"]: concept_b}

    for mean_method in MEANS:
        guide = Concept.average([concept_a, concept_b], method=mean_method)
        for lam in LAMBDAS:
            torch.cuda.empty_cache()
            texts, probe = fur.generate_with_topk_beam_guide(
                PROMPTS,
                concepts=[guide],
                k=K,
                steps=STEPS,
                fluency_weight=lam,
                batch_size=4,
            )
            harness.evaluate(
                PROMPTS,
                texts,
                synsets,
                config={
                    "stratum": stratum,
                    "rho": info["rho"],
                    "mean": mean_method,
                    "lam": lam,
                    "k": K,
                    "steps": STEPS,
                },
                probe=probe,
                baseline_texts=baseline_texts,
                concepts=concepts,
            )
            print(f"{stratum} / {mean_method} / lam={lam:g}: done")

In [ ]:
parsed = [json.loads(line) for line in LOG_PATH.read_text().strip().split("\n")]
assert len(parsed) == 3 * len(MEANS) * len(LAMBDAS) * len(
    PROMPTS
), "one record per cell per prompt"
assert all(r["expression"] is not None for r in parsed), "metric v2 present everywhere"
print(f"gate: {len(parsed)} records parse, expression logged")

In [ ]:
rows = []
for record in parsed:
    config = record["config"]
    info = SELECTION[config["stratum"]]
    expr = record["expression"]
    rows.append(
        {
            "stratum": config["stratum"],
            "rho": config["rho"],
            "mean": config["mean"],
            "lam": config["lam"],
            "prompt": record["prompt"],
            "concept_a": info["a"],
            "concept_b": info["b"],
            "delta_a": expr[info["a"]]["delta"],
            "delta_b": expr[info["b"]]["delta"],
            "ppl_ratio": record["ppl_ratio"],
            "fluency_flag": record["fluency_flag"],
        }
    )
results = pd.DataFrame(rows)

for side in ("a", "b"):
    zs = []
    for concept, group in results.groupby(f"concept_{side}")[f"delta_{side}"]:
        z = (group - group.mean()) / (group.std() + 1e-8)
        zs.append(z)
    results[f"z_{side}"] = pd.concat(zs).sort_index()
results["joint_z"] = results[["z_a", "z_b"]].min(axis=1)

summary = (
    results.groupby(["stratum", "rho", "mean", "lam"])
    .agg(
        delta_a=("delta_a", "mean"),
        delta_b=("delta_b", "mean"),
        joint_z=("joint_z", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        flag_share=("fluency_flag", "mean"),
    )
    .reset_index()
    .sort_values(["rho", "mean", "lam"])
)
summary.to_csv("resources/21_e1b_summary_unquantized.csv", index=False)
summary.round(4)

## Point 2 first: does the aligned mean stack with beam search?

Paired per-prompt joint-z gap (aligned − extrinsic) at each λ and stratum.

In [ ]:
wide = results.pivot_table(
    index=["stratum", "rho", "lam", "prompt"], columns="mean", values="joint_z"
).reset_index()
wide["gap"] = wide["aligned"] - wide["extrinsic"]
gaps = (
    wide.groupby(["stratum", "rho", "lam"])["gap"]
    .agg(["mean", "sem"])
    .reset_index()
    .sort_values(["rho", "lam"])
)
print(gaps.round(4).to_string(index=False))

## The dial: λ-trajectories on the (fluency, expression) plane

Each line is one mean's λ-trajectory within a stratum (λ=0 at the top-right end). The gray dashed line is the 2.5× PPL guardrail; the × marks the best greedy method from E1 for scale.

In [ ]:
E1_BEST_GREEDY = {  # from resources/20_e1_summary_unquantized.csv, for visual reference
    "low": (7.50, -0.106),
    "medium": (6.41, -0.183),
    "high": (7.58, -0.122),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True)
colors = {"extrinsic": "tab:blue", "aligned": "tab:green"}

for ax, (stratum, info) in zip(axes, SELECTION.items()):
    cell = summary[summary["stratum"] == stratum]
    for mean_method in MEANS:
        traj = cell[cell["mean"] == mean_method].sort_values("lam")
        ax.plot(
            traj["ppl_ratio"],
            traj["joint_z"],
            "o-",
            color=colors[mean_method],
            label=mean_method,
        )
        for _, row in traj.iterrows():
            ax.annotate(
                f"lam={row['lam']:g}",
                (row["ppl_ratio"], row["joint_z"]),
                textcoords="offset points",
                xytext=(6, -3),
                fontsize=7,
            )
    ax.plot(
        *E1_BEST_GREEDY[stratum],
        "x",
        color="black",
        markersize=9,
        label="best greedy (E1)",
    )
    ax.axvline(2.5, color="gray", ls="--", lw=0.8)
    ax.set_title(f"{stratum} (rho={info['rho']:.2f})")
    ax.set_xlabel("mean PPL ratio")
axes[0].set_ylabel("mean joint z")
axes[0].legend(fontsize=8)

plt.tight_layout()
plt.savefig("resources/21_e1b_lambda_dial_unquantized.png", dpi=150)
plt.show()

In [ ]:
def answer(text: str) -> str:
    return text.split("assistant<|end_header_id|>")[-1]


for stratum in SELECTION:
    for lam in LAMBDAS:
        example = next(
            r
            for r in parsed
            if r["config"]["stratum"] == stratum
            and r["config"]["mean"] == "extrinsic"
            and r["config"]["lam"] == lam
            and "short story" in r["prompt"]
        )
        print(
            f"[{stratum} / extrinsic / lam={lam:g}] ppl_ratio={example['ppl_ratio']:.1f}"
        )
        print("   ", answer(example["text"])[:130])
    print()